# QLoRA fine-tune: Qwen2.5-1.5B-Instruct on text-to-SQL

Made for a free Colab T4 (16GB). Base model loads in 4-bit NF4, LoRA adapters train on top. One epoch over 8k examples takes roughly 2-3 hours.

Runtime > Change runtime type > T4 GPU before running anything.

In [ ]:
!pip install -q transformers==4.48.0 trl==0.13.0 peft==0.14.0 bitsandbytes==0.45.0 accelerate==1.2.1 datasets==3.2.0

In [ ]:
import os

# grab the processed data from the repo (skip if already cloned)
if not os.path.exists("text2sql-qlora"):
    !git clone https://github.com/Akshu24Tech/text2sql-qlora.git

TRAIN_PATH = "text2sql-qlora/data/train.jsonl"

import torch
print(torch.cuda.get_device_name(0))

In [ ]:
from datasets import load_dataset

train_ds = load_dataset("json", data_files=TRAIN_PATH, split="train")
print(train_ds)
train_ds[0]

## Prompt format

Each example becomes a short chat: system prompt fixes the task, user turn carries the schema + question, assistant turn is the target SQL. Using the model's own chat template so inference later looks exactly like training.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

SYSTEM = "You are a text-to-SQL assistant. Given a table schema and a question, reply with only the SQL query, nothing else."

def to_chat_text(row):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"},
        {"role": "assistant", "content": row["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = train_ds.map(to_chat_text, remove_columns=train_ds.column_names)
print(train_ds[0]["text"])

In [ ]:
# sanity check token lengths against the 512 budget
lens = [len(tokenizer(t)["input_ids"]) for t in train_ds["text"][:1000]]
import numpy as np
print(f"p50={int(np.percentile(lens, 50))}  p95={int(np.percentile(lens, 95))}  max={max(lens)}")

## Model in 4-bit + LoRA config

NF4 quantization with fp16 compute (T4 has no bf16). LoRA r=16 on all attention and MLP projections, which worked noticeably better than attention-only in a quick smoke test.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False  # no kv cache during training

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="qwen25-sql-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_seq_length=512,
    dataset_text_field="text",
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=25,
    save_strategy="epoch",
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.model.print_trainable_parameters()

In [ ]:
trainer.train()

## Save the adapter

Only the LoRA weights get saved (~70MB), not the full model. Pushed to HF Hub so the eval notebook can pull them without re-uploading anything.

In [ ]:
trainer.save_model("qwen25-sql-lora/final")

# needs HF_TOKEN set in Colab secrets (write access)
from huggingface_hub import login
from google.colab import userdata
login(userdata.get("HF_TOKEN"))

trainer.model.push_to_hub("Akshu24Tech/qwen25-1.5b-sql-lora")
tokenizer.push_to_hub("Akshu24Tech/qwen25-1.5b-sql-lora")

In [ ]:
# quick vibe check before proper eval
from transformers import pipeline

sample = {
    "context": "CREATE TABLE employees (name VARCHAR, department VARCHAR, salary INTEGER)",
    "question": "What is the average salary in the engineering department?",
}
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": f"Schema:\n{sample['context']}\n\nQuestion: {sample['question']}"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = trainer.model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))